# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and includes data on protein abundance, modifications, and sequences from human (Homo sapiens) samples. All dataset elements (record sets, fields, columns) are referenced by their `@id` fields.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Access the dataset metadata attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")print(f"Authors: {getattr(metadata, 'author', None)}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print all available record sets (`@id`), their labels, and for each their corresponding fields (`@id`, name, and data type).

In [ ]:
# Explore RecordSets, Fields, and Columns by their @id
def print_record_sets(dataset):
    record_sets = getattr(dataset, 'record_sets', None)
    if not record_sets:
        # Sometimes as in this schema, record sets may be available directly as a metadata attribute.
        record_sets = getattr(dataset.metadata, 'recordSets', None) or getattr(dataset.metadata, 'recordSet', None) or []

    print('Record Sets (@id):')
    if not record_sets or len(record_sets) == 0:
        print('No record sets found in schema.')
    else:
        if isinstance(record_sets, dict):
            record_sets = [record_sets]  # Ensure it's a list
        for rs in record_sets:
            rec_id = getattr(rs, '@id', None) or rs.get('@id')
            label = getattr(rs, 'label', None) or rs.get('label')
            print(f"- {rec_id} (label: {label})")
            # List fields for the record set
            fields = getattr(rs, 'fields', None) or rs.get('field')
            if fields:
                if isinstance(fields, dict):
                    fields = [fields]
                for field in fields:
                    field_id = getattr(field, '@id', None) or field.get('@id')
                    name = getattr(field, 'name', None) or field.get('name')
                    data_type = getattr(field, 'dataType', None) or field.get('dataType')
                    print(f"    - Field @id: {field_id}, name: {name}, dataType: {data_type}")
            print()

print_record_sets(dataset)

Based on most Croissant schemas, you can explicitly enumerate available record sets and their IDs using `dataset.record_sets` or metadata. For this dataset, if no explicit record sets exist in the schema, we'll attempt to enumerate possible record set `@id`s.

In [ ]:
# List available record sets by @id
record_sets = []
try:
    record_sets = dataset.record_sets
except Exception:
    # fallback: Try to get from metadata
    record_sets = getattr(metadata, 'recordSet', [])
if isinstance(record_sets, dict):
    record_sets = [record_sets]

print('Available record set @id values:')
for rset in record_sets:
    # rset is a dict or CroissantEntity
    rsid = None
    if hasattr(rset, '@id'):
        rsid = getattr(rset, '@id')
    elif isinstance(rset, dict):
        rsid = rset.get('@id')
    print(rsid)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
For this notebook, we will guess the most likely record set(s) if not explicitly listed in the schema - typically, the main data table is the primary record set. You should specify the desired record set's `@id`.

In [ ]:
# For this schema, we'll try the primary record set, if available. Otherwise, guess from distributions.
main_record_set_id = None  # e.g., 'cr:protein_table' (this is an illustrative example)

if record_sets and isinstance(record_sets[0], dict):
    main_record_set_id = record_sets[0].get('@id')
elif record_sets and hasattr(record_sets[0], '@id'):
    main_record_set_id = getattr(record_sets[0], '@id')

# If not found, try to guess from the known usual structures (distribution file may correspond to the main record set)
if not main_record_set_id:
    # Try to fetch the ID from 'distribution' as fallback
    dist = getattr(metadata, 'distribution', None)
    if dist:
        if isinstance(dist, list):
            main_record_set_id = dist[0].get('@id') if isinstance(dist[0], dict) else None
        elif isinstance(dist, dict):
            main_record_set_id = dist.get('@id')
# Safety check
if not main_record_set_id:
    print("No valid record set id found. Please refer to the schema to determine available record sets.")
else:
    print(f"Using record set @id: {main_record_set_id}")
    # Load records and into dataframe
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    print(f"Columns in DataFrame: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Start by identifying a numeric field (`@id`) for analysis. We'll filter, normalize, and group data by another field, using their `@id`s.

In [ ]:
# For demonstration, we'll try to pick probable numeric and group fields by their likely @id.
if main_record_set_id and not df.empty:
    # Example: Try common protein table columns
    # You should adjust 'MW' and 'Gene_Symbol' (or similar) to valid column names reflected in df.columns, using their @id in a real scenario
    import numpy as np

    print('Available columns:')
    print(df.columns.tolist())

    # Manually set field IDs as seen in df.columns for demonstration purposes
    # For this dataset, suppose 'MW' refers to molecular weight (numeric), 'Gene_Symbol' to gene group
    numeric_field_id = None
    group_field_id = None

    for col in df.columns:
        if 'MW' in col or 'mol' in col.lower() or 'weight' in col.lower():
            numeric_field_id = col
        if 'Gene' in col or 'symbol' in col.lower() or 'gene_symbol' in col.lower():
            group_field_id = col
    if not numeric_field_id:
        numeric_field_id = df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) > 0 else df.columns[0]
    if not group_field_id:
        group_field_id = df.columns[0]
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Remove non-numeric values if there are parse errors
    try:
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    except Exception:
        pass

    threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use matplotlib/seaborn for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not df.empty and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        # Show top groups by mean value
        top_groups = grouped_df.head(10)
        plt.figure(figsize=(10,5))
        sns.barplot(data=top_groups, x=group_field_id, y=numeric_field_id)
        plt.title(f'Mean {numeric_field_id} by {group_field_id} (Top 10)')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We've illustrated how to load, explore, and perform basic analysis and visualization of a mass spectrometry protein dataset using `mlcroissant`. You can adapt the field `@id` values to your specific schema for more targeted exploration. 

Key findings would depend on your research questions–for example, discovering which proteins show highest molecular weight, abundance, or grouping trends by genes. For downstream machine learning or statistical analysis, continue refining preprocessing and feature selection as needed.

---
For more on the FAIR² dataset, see [FAIR² record](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).